# Análisis Exploratorio de Datos: Elecciones al Senado de Colombia 2018
### Notebook pedagógico — Curso de Analítica de Datos y Machine Learning en Ciencias Políticas
**Universidad de Antioquia** — Facultad de Derecho y Ciencias Políticas — 2026-2

---

#### ¿Qué es un EDA y por qué lo hacemos?

Un **Análisis Exploratorio de Datos (EDA)** es el proceso sistemático de examinar un dataset antes de formular modelos o realizar inferencias. El concepto fue introducido por el estadístico John Tukey en 1977 y sigue siendo la piedra angular de cualquier proyecto serio de ciencia de datos.

Piensa en el EDA como la exploración de un territorio desconocido: necesitamos recorrerlo, observar su geografía y sus caminos antes de trazar un mapa. En nuestro caso, el «territorio» son los **resultados electorales al Senado de la República de Colombia en las elecciones de 2018**, provenientes del CEDAE de la Registraduría Nacional del Estado Civil.

**¿Qué contiene el dataset?** Los votos obtenidos por cada partido y cada candidato en cada municipio del país, tanto para la **circunscripción Nacional** como para la **Indígena**.

**Preguntas que guiarán nuestra exploración:**
- ¿Cómo se distribuye el voto entre partidos y candidatos?
- ¿Hay unos pocos candidatos que concentran la mayoría de los votos, o la competencia es pareja?
- ¿Qué diferencias existen entre la circunscripción Nacional y la Indígena?
- ¿La competencia electoral varía significativamente entre departamentos?
- ¿Qué variables podrían predecir si un candidato obtiene curul?

**Estructura del notebook:** Seguiremos un roadmap de 23 pasos organizados en tres niveles:
- 🟢 **Básico** (Pasos 1–6): Auditoría estructural y panorama inicial
- 🟡 **Intermedio** (Pasos 7–14): Limpieza, interacciones y contexto
- 🔴 **Avanzado** (Pasos 15–23): Estructura latente y preparación para modelado

> 💡 **Ojito:** Cada paso comienza con una explicación de *qué* se va a hacer y *por qué*. Lee las celdas de texto antes de ejecutar el código — entender la lógica es tan importante como ejecutar las instrucciones.

---
# 🟢 Nivel Básico — Auditoría Estructural y Panorama Inicial

**Objetivo:** ¿Qué tenemos entre manos, cuál es la forma de los datos y en qué estado de calidad se encuentran?

En este nivel vamos a conocer el dataset como si fuera un nuevo vecindario: recorrer sus calles, contar sus casas, verificar que las direcciones sean correctas y tomar nota de lo que se ve a primera vista.

## Paso 1 · Carga y primer contacto
🟢 Básico

Antes de cualquier análisis, necesitamos **cargar los datos y mirarlos**. Así como un politólogo empieza su investigación leyendo las fuentes primarias, un analista de datos empieza leyendo sus datos.

Vamos a:
1. Importar el archivo CSV con `pandas`
2. Ver las primeras y últimas filas (`.head()`, `.tail()`)
3. Tomar una muestra aleatoria (`.sample()`)
4. Verificar las dimensiones: ¿cuántas filas y columnas tiene?

> **Pregunta guía:** ¿Cuántos registros tiene el dataset y qué representa cada fila? ¿Es un registro por candidato, por partido, por municipio, o una combinación?

### 🔍 Interpretación — Paso 1

El dataset contiene más de **300,000 registros** y **16 columnas**. Cada fila representa una combinación única de **municipio × partido o candidato**. Esto quiere decir que un mismo candidato (por ejemplo, Álvaro Uribe) aparece **una vez por cada municipio** del país donde está registrada la votación.

Hay dos tipos de filas:
- **Filas de partido** (`codigo_lista = 0`): consolidan los votos de lista cerrada del partido en ese municipio. En la columna `nombres` aparece el nombre del partido.
- **Filas de candidato** (`codigo_lista ≥ 1`): votos individuales de cada candidato en ese municipio. La columna `nombres` contiene el primer nombre del candidato, y `primer_apellido` / `segundo_apellido` completan la identificación.

Esta estructura «granular» es muy rica porque permite análisis a múltiples niveles (candidato, partido, municipio, departamento), pero implica que debemos tener **cuidado de no mezclar los dos tipos de filas** al calcular estadísticas.

## Paso 2 · Estructura y tipos de datos
🟢 Básico

Cada columna de un dataset tiene un **tipo de dato** asignado automáticamente al cargar el archivo. Un error muy frecuente es que el sistema asigne un tipo incorrecto.

> **Analogía política:** El código DANE de Antioquia es `5` y el de Bogotá es `11`. ¿Tiene sentido calcular el *promedio* de esos códigos? ¡Por supuesto que no! Son **etiquetas categóricas** disfrazadas de números. Sumarlos o promediarlos produciría un resultado sin sentido.

> **Pregunta guía:** ¿Los tipos de datos asignados automáticamente son correctos? ¿Hay variables categóricas mal clasificadas como numéricas?

### 🔍 Interpretación — Paso 2

Detectamos un problema clásico: los **códigos DANE** (`coddpto`, `codmpio`) fueron cargados como enteros. Lo mismo ocurrió con `codigo_partido` e `id_electoral`.

Estos campos no son cantidades sino **identificadores categóricos**. Si los dejáramos como enteros:
- Un gráfico podría ordenar los departamentos «numéricamente» en lugar de por nombre.
- Un modelo de ML los trataría como variable continua y asumiría que departamento «15» está «más lejos» de «5» que de «8».

> **Lección general:** Siempre verifica los tipos de datos al cargar un dataset. Los sistemas automáticos no conocen el contexto — esa es tu responsabilidad como analista.

## Paso 3 · Diagnóstico de valores faltantes
🟢 Básico

Los **valores faltantes** (nulos, `NaN`) son una realidad en prácticamente cualquier dataset. Pero no todos los faltantes son iguales. En estadística distinguimos tres tipos:

| Tipo | Significado | Ejemplo |
|------|-------------|---------|
| **MCAR** (*Missing Completely At Random*) | El dato falta por razones completamente aleatorias | Se cayó una hoja y se perdieron datos al azar |
| **MAR** (*Missing At Random*) | Falta dependiendo de *otra* variable observable | Municipios pequeños reportan menos datos |
| **MNAR** (*Missing Not At Random*) | Falta por razones que dependen del *propio dato* faltante | El dato simplemente no debería existir para esa fila |

> **Pregunta guía:** ¿Hay valores nulos en el dataset? ¿Son aleatorios o responden a un patrón estructural?

### 🔍 Interpretación — Paso 3

Los valores nulos aparecen **exclusivamente** en las columnas `primer_apellido` y `segundo_apellido`, y coinciden casi perfectamente con las filas de partido (`codigo_lista = 0`).

Esto **no es un error de recolección** sino un diseño estructural del dataset:
- Las filas de **partido** no tienen persona asociada → `primer_apellido` y `segundo_apellido` están vacíos.
- Las filas de **candidato** (`codigo_lista ≥ 1`) siempre tienen nombre.

En la terminología de valores faltantes, esto es **MNAR** (*Missing Not At Random*): los datos faltan porque *no deberían existir*. No hay un nombre que «se perdió» — simplemente no aplica.

> **Decisión:** No vamos a imputar estos nulos. Son consecuencia de la estructura del dataset donde coexisten dos niveles (partido y candidato). Cuando sea necesario, trabajaremos con subconjuntos filtrados por `codigo_lista`.

## Paso 4 · Diagnóstico de duplicados y rangos plausibles
🟢 Básico

Este paso es la **primera línea de defensa** contra datos corruptos. Un solo duplicado puede distorsionar un promedio; un voto negativo puede sabotear una gráfica.

Vamos a verificar:
1. ¿Hay filas completamente duplicadas?
2. ¿Hay duplicados por **clave lógica** (`id_electoral` + `codigo_partido` + `codigo_lista`)?
3. ¿Los valores numéricos están en rangos razonables? (votos ≥ 0, curules ∈ {0,1}, etc.)

> **Pregunta guía:** ¿Podemos confiar en que cada fila es un registro único y que los valores son plausibles?

### 🔍 Interpretación — Paso 4

**Buenas noticias:**
- **No hay duplicados** por clave lógica — cada combinación municipio-partido-posición es única.
- **No hay votos negativos** ni curules con valores imposibles.
- Los datos corresponden exclusivamente al **año 2018** y a la elección de **Senado**.

Hay candidatos con **0 votos** en muchos municipios; esto no es un error. En el voto preferente colombiano, un candidato aparece en la tarjeta electoral de todos los municipios pero solo recibe votos en algunos. Los ceros son información legítima.

> **Lección:** En datos oficiales de la Registraduría la calidad suele ser buena. En datasets de menor calidad (encuestas, scraping, registros administrativos locales), este paso suele revelar sorpresas desagradables.

## Paso 5 · Estadística descriptiva univariada
🟢 Básico

La **estadística descriptiva** nos da un resumen numérico de cada variable. Las medidas clave son:
- **Tendencia central:** media (promedio), mediana (valor central), moda (valor más frecuente)
- **Dispersión:** desviación estándar, rango intercuartílico (IQR), mínimo y máximo
- Para categóricas: frecuencias, cardinalidad (número de categorías únicas)

⚠️ **Importante:** Debemos separar las filas de partido (`codigo_lista = 0`) de las de candidato (`codigo_lista ≥ 1`). Mezclarlas distorsionaría los promedios porque son unidades de análisis diferentes.

> **Pregunta guía:** ¿Cuántos votos obtiene un candidato típico en un municipio? ¿Y un partido? ¿La media y la mediana coinciden o difieren mucho?

### 🔍 Interpretación — Paso 5

Los valores descriptivos revelan un patrón fundamental:

- **La media de votos por candidato superá ampliamente a la mediana.** Esto indica una distribución **fuertemente sesgada a la derecha**: la gran mayoría obtiene pocos votos mientras un puñado de figuras concentra cifras enormes.
- La **asimetría (skewness)** es muy alta (>>0), confirmando el sesgo. Una distribución simétrica tendría skewness ≈ 0.
- La **moda es 0**: el valor más frecuente es cero votos, lo cual tiene sentido político — candidatos de listas pequeñas aparecen en todos los municipios pero solo reciben votos en algunos.

> **Reflexión política:** Esta distribución refleja la estructura del sistema electoral colombiano. Con decenas de partidos y cientos de candidatos compitiendo por 108 curules, la competencia sigue un patrón de «pocos ganadores y muchos perdedores». Veremos esto con más claridad en las visualizaciones.

## Paso 6 · Visualización univariada y detección de outliers
🟢 Básico

Las tablas numéricas del paso anterior dieron una idea general, pero una buena **visualización** revela patrones que los números solos no muestran. Crearemos:

- **Histogramas** para ver la forma de la distribución de votos
- **Boxplots** para visualizar la dispersión y detectar **outliers** (valores atípicos)
- **Gráficos de barras** para las variables categóricas principales

Un **outlier** es una observación inusualmente alejada del resto. No siempre es un error — a veces es la información más importante.

> **Pregunta guía:** ¿La distribución de votos es simétrica o hay unos pocos candidatos que concentran la mayoría? ¿Quiénes son los outliers y por qué lo son?

### 🔍 Interpretación — Paso 6

La distribución de votos por candidato es **extremadamente sesgada**. La gran mayoría de registros tiene pocos votos (muchos son cero), mientras unos pocos candidatos acumulan cifras enormes.

Los **outliers** más extremos no son errores de registro — son los **senadores más votados del país**. Eliminarlos sería eliminar la información más importante del dataset.

> **Decisión clave:** No eliminamos estos outliers. En datos electorales, los valores extremos *son* la historia: representan a los políticos con mayor capital electoral. Un análisis que los excluya estaría literalmente borrando a los actores más relevantes.

El contraste entre la circunscripción Nacional y la Indígena en volumen de votos ya es visible. Lo exploraremos con mayor profundidad adelante.

---
# 🟡 Nivel Intermedio — Limpieza, Interacciones y Contexto

**Objetivo:** ¿Cómo estandarizo la información y cómo se relacionan las variables entre sí para revelar verdaderos patrones?

En este nivel dejamos de solo *describir* los datos y empezamos a *interrogarlos*: buscamos relaciones entre variables, limpiamos inconsistencias y contextualizamos los hallazgos.

## Paso 7 · Tratamiento de valores faltantes e inconsistencias
🟡 Intermedio

Ahora que conocemos la estructura del dataset, debemos **estandarizarlo**. En datos de texto libre (nombres de municipios, partidos), es frecuente encontrar inconsistencias tipográficas: tildes faltantes, mayúsculas irregulares, espacios extra.

Vamos a:
1. **Documentar las decisiones** sobre nulos (ya resueltas en el Paso 3).
2. **Estandarizar nombres** de municipios y partidos para evitar que «MEDELLIN» y «Medellín» cuenten como dos entidades distintas.
3. **Crear una columna de nombre de partido** reutilizable para todo el análisis.

> **Pregunta guía:** ¿Hay inconsistencias en los nombres que podrían causar errores al agrupar datos?

### 🔍 Interpretación — Paso 7

El dataset original ya viene en **mayúsculas sin tildes**, que es el formato estándar de la Registraduría. Mantuvimos este formato por consistencia. Encontramos algunos partidos con espacios dobles internos (como `"MOVIMIENTO AUTORIDADES INDIGENAS DE COLOMBIA  AICO"`) que corregimos al crear el mapa de partidos.

Decisiones clave:
- **Nulos en `primer_apellido`/`segundo_apellido`:** No se imputan (son estructurales, MNAR).
- **Formato de texto:** Se mantiene en mayúsculas sin tildes, coherente con la fuente original.
- **Nuevas columnas:** `nombre_partido` (para todas las filas) y `nombre_candidato` (solo candidatos), que facilitan el análisis posterior.

> Los nombres de partidos como `"COALICION LISTA DE LA DECENCIA (ASI;UP;MAIS)"` reflejan las coaliciones electorales colombianas: varios movimientos que se presentan juntos bajo una sola lista.

## Paso 8 · Tratamiento de outliers
🟡 Intermedio

En el Paso 6 identificamos outliers en la distribución de votos. Ahora debemos decidir **qué hacer con ellos**, documentando nuestra justificación.

El método estadístico más común para detectar outliers es la **regla del IQR** (Rango Intercuartílico):
- Se calcula: `IQR = Q3 - Q1`
- Todo valor por debajo de `Q1 - 1.5 × IQR` o por encima de `Q3 + 1.5 × IQR` se marca como outlier.

Pero **cuidado**: ser un outlier estadístico no significa ser un dato erróneo.

> **Pregunta guía:** ¿Cuántos registros caen como "outlier estadístico" según la regla IQR? ¿Se trata de errores o de información legítima y valiosa?

### 🔍 Interpretación — Paso 8

La regla del IQR clasifica una proporción notable de registros como "outliers". Pero examinando estos casos, vemos que incluyen a **los candidatos más votados del país**: senadores que obtuvieron su curul precisamente por esos votos.

> **Decisión: MANTENER todos los outliers.**
>
> **Justificación:**
> 1. No son errores de registro — son datos verificados por la Registraduría.
> 2. Son legítimos y sustantivamente importantes — representan a los senadores electos.
> 3. Eliminarlos sesgaría el análisis al borrar la información más relevante.
> 4. La distribución sesgada es una propiedad *real* de la competencia electoral, no un artefacto de los datos.

Esta es una lección importante: **las herramientas estadísticas no tienen contexto**. La regla del IQR es útil para detectar errores de captura en datos industriales, pero en datos electorales un "outlier" puede ser el presidente de la República. El analista debe aportar el juicio que la fórmula no puede.

## Paso 9 · Análisis bivariado: numérica vs. numérica
🟡 Intermedio

Hasta ahora hemos examinado una variable a la vez (análisis **univariado**). Ahora buscamos **relaciones entre pares de variables numéricas**. La herramienta principal es la **correlación**: un número entre -1 y +1 que mide la fuerza de la relación lineal.

⚠️ **Correlación ≠ Causalidad.** Que dos variables se muevan juntas no significa que una cause la otra.

Para tener variables numéricas comparables, agregaremos los datos a nivel **partido × departamento**, calculando: total de votos del partido y número de candidatos presentados.

> **Pregunta guía:** ¿Los partidos que presentan más candidatos obtienen más votos? ¿O la cantidad de candidatos no predice el éxito electoral?

### 🔍 Interpretación — Paso 9

La correlación entre el número de candidatos y los votos totales nos dice si «presentar más candidatos» se traduce en más votos. La correlación positiva indica que los partidos más grandes tienden tanto a presentar más candidatos como a obtener más votos — pero esto no significa que presentar muchos candidatos *cause* más votos. Más bien refleja que los partidos con mayor estructura organizativa tienen **ambas cosas a la vez**.

> **Reflexión:** En ciencia política, esto se relaciona con el concepto de **estructura partidista**. Los partidos con mayor implantación territorial presentan más candidatos *y* movilizan más electores. La variable latente es la fortaleza organizativa del partido.

## Paso 10 · Análisis bivariado: categórica y mixta
🟡 Intermedio

Ahora cruzamos variables **categóricas** entre sí (tablas de contingencia) y categóricas con numéricas (boxplots agrupados). Esto nos permite responder preguntas como: ¿la distribución de votos varía entre departamentos? ¿Qué partidos dominan en cada circunscripción?

> **Pregunta guía:** ¿Cómo varía la distribución de votos entre los departamentos más grandes del país? ¿Qué partidos dominan la circunscripción Nacional vs. la Indígena?

### 🔍 Interpretación — Paso 10

El análisis bivariado revela diferencias territoriales significativas:

- Los departamentos con mayor población (Bogotá, Antioquia, Valle del Cauca) tienen, como es esperable, más votos por candidato. Pero la **dispersión** interna es enorme: incluso en Antioquia, la mayoría de candidatos reciben pocos votos.
- La tabla de contingencia muestra que los grandes partidos concentran casi toda su votación en la circunscripción **Nacional**, mientras que la circunscripción **Indígena** tiene partidos completamente diferentes (movimientos étnicos y comunitarios).

> **Reflexión política:** Las dos circunscripciones son esencialmente **elecciones paralelas** con actores distintos. Analizarlas como un solo bloque enmascararía estas diferencias fundamentales.

## Paso 11 · Análisis de distribuciones
🟡 Intermedio

En el Paso 6 vimos que la distribución de votos es extremadamente sesgada. Ahora vamos a **diagnosticar formalmente** esa distribución y a aplicar una **transformación logarítmica** para hacerla más simétrica.

**¿Por qué importa la forma de la distribución?**
- Muchos tests estadísticos y modelos de ML asumen o se benefician de distribuciones simétricas (cercanas a la «normal» o campana de Gauss).
- La transformación logarítmica es particularmente adecuada para datos que siguen una **distribución log-normal**: fenómenos donde hay muchos valores pequeños y pocos valores muy grandes. Esto es típico en competencia electoral, ingresos económicos, tamaños de ciudades, etc.

Un **QQ-plot** (*Quantile-Quantile plot*) compara los cuantiles de nuestros datos contra los cuantiles teóricos de una distribución normal. Si los puntos caen sobre la línea diagonal, la distribución es normal.

> **Pregunta guía:** ¿La distribución de votos es normal? Y después de la transformación logarítmica, ¿se acerca más?

### 🔍 Interpretación — Paso 11

La transformación logarítmica **reduce drásticamente la asimetría** y produce una distribución mucho más cercana a la normal:
- La asimetría pasa de un valor muy alto a uno cercano a cero.
- El QQ-plot de los datos log-transformados se alinea mucho mejor con la recta de referencia (distribución normal teórica).

Esto confirma que los votos siguen aproximadamente una **distribución log-normal**, que es típica en fenómenos de competencia donde hay:
- **Rendimientos crecientes**: los candidatos conocidos atraen más atención, lo cual atrae más votos (efecto «bola de nieve»).
- **Restricciones de no-negatividad**: los votos no pueden ser negativos.

> **Reflexión:** La distribución log-normal aparece en muchos fenómenos sociales: ingresos, tamaño de ciudades, citas académicas, número de seguidores en redes sociales. Todos comparten una estructura de «pocos concentran mucho». En elecciones, esto refleja la **ley de hierro de la oligarquía** de Michels: la competencia tiende naturalmente a la concentración.

## Paso 12 · Segmentación por subgrupos
🟡 Intermedio

Este es uno de los pasos más importantes del EDA: **¿los patrones generales se sostienen cuando desagregamos por subgrupos?**

La **paradoja de Simpson** es un fenómeno estadístico donde una tendencia que existe en los datos agregados **desaparece o se invierte** cuando se analiza dentro de subgrupos. Es como si las estadísticas nacionales de empleo mostraran mejora, pero al mirar región por región todas estuvieran empeorando.

Vamos a segmentar por la variable más relevante: **circunscripción Nacional vs. Indígena**.

> **Pregunta guía:** ¿Las conclusiones que sacamos del dataset completo se sostienen cuando separamos las dos circunscripciones? ¿Hay algún patrón que cambie o se invierta?

### 🔍 Interpretación — Paso 12

La segmentación revela que las dos circunscripciones son **mundos electorales completamente distintos**:

| Aspecto | Nacional | Indígena |
|---------|----------|----------|
| Volumen de votos | Cientos de miles por candidato top | Decenas de miles |
| Partidos dominantes | Centro Democrático, Cambio Radical, etc. | AICO, MAIS, Alianza Social Independiente |
| Concentración del voto | Alta (pocos candidatos muy votados) | Más dispersa |
| Proporción de ceros | Alta | Muy alta |

Si hubiéramos analizado el dataset sin separar circunscripciones, los patrones de la Nacional (que tiene muchas más filas) habrían **enmascarado completamente** la dinámica de la Indígena.

> **Reflexión política:** La circunscripción especial indígena fue creada por la Constitución de 1991 para garantizar representación de los pueblos indígenas en el Senado (2 curules de las 108). Tiene partidos, dinámicas y escalas de votación completamente diferentes. Analizarla con los mismos parámetros que la Nacional sería un error analítico y político.

## Paso 13 · Exploración temporal
🟡 Intermedio — ⚠️ NO APLICA A ESTE DATASET

Este paso analiza **tendencias, estacionalidad y quiebres en el tiempo**. Sin embargo, nuestro dataset contiene datos de **una sola elección** (Senado 2018), por lo que no hay dimensión temporal que explorar.

**¿Cuándo se activaría este paso?** Si tuviéramos datos de múltiples elecciones (por ejemplo, Senado 2006, 2010, 2014, 2018, 2022), podríamos analizar:
- ¿Cómo ha evolucionado la concentración del voto?
- ¿Hay partidos que crecen o declinan sistemáticamente?
- ¿El número efectivo de partidos aumenta o disminuye con las reformas electorales?
- ¿La participación electoral tiene tendencia o es cíclica?

> 💡 **Ejercicio sugerido para el estudiante:** Descarga datos de elecciones anteriores del portal de la Registraduría y construye un análisis temporal comparado.

## Paso 14 · Exploración geoespacial
🟡 Intermedio

Colombia tiene 32 departamentos y más de 1,100 municipios. Las dinámicas electorales suelen tener un fuerte componente territorial: los partidos tienen bastiones regionales, y la participación varía enormemente entre zonas urbanas y rurales.

Lo ideal sería crear un **mapa coroplético** (un mapa donde cada departamento/municipio se colorea según una variable). Esto requiere la librería `geopandas` y un archivo de geometrías (shapefile) de Colombia. Como alternativa práctica, crearemos un **gráfico de barras horizontal** ordenado que funcione como «mapa conceptual» del territorio.

> **Pregunta guía:** ¿Cómo se distribuyen los votos geográficamente? ¿Hay departamentos que concentran desproporcionadamente la votación?

### 🔍 Interpretación — Paso 14

La distribución geográfica del voto muestra una **fuerte concentración territorial**:
- Los 5 departamentos más poblados (Bogotá, Antioquia, Valle del Cauca, Atlántico, Santander) concentran una proporción muy significativa del total de votos al Senado.
- Departamentos como Vaupés, Guainía, Vichada y Amazonas tienen volúmenes de votación mínimos — coherente con sus poblaciones pequeñas.

> **Reflexión política:** Esta concentración geográfica del voto tiene implicaciones profundas para la representación: los candidatos al Senado necesitan ser competitivos en los grandes centros urbanos para ganar curul. Esto puede marginar las agendas de los departamentos periféricos, que terminan subrepresentados en las prioridades legislativas.

---
# 🔴 Nivel Avanzado — Estructura Latente y Preparación para el Modelo

**Objetivo:** ¿Qué patrones invisibles existen en los datos, hay sesgos en la recolección y cómo adecuamos los datos para el Machine Learning?

En este nivel usamos herramientas más sofisticadas: creación de nuevas variables, reducción de dimensionalidad, clustering, tests de hipótesis y evaluación de sesgos.

## Paso 15 · Ingeniería de características (Feature Engineering)
🔴 Avanzado

**Ingeniería de características** (o *Feature Engineering*) es el proceso de crear nuevas variables a partir de las existentes para capturar patrones más complejos. Es como construir indicadores compuestos: el PIB per cápita es más informativo que el PIB solo o la población sola.

Vamos a crear variables a nivel de **municipio** que capturen diferentes dimensiones de la competencia electoral:

| Variable | Significado |
|----------|-------------|
| `votos_pct` | % de votos de cada partido sobre el total del municipio |
| `n_partidos_municipio` | Número de partidos que compitieron |
| `hhi_municipio` | Índice Herfindahl-Hirschman: concentración del voto |
| `tiene_circ_indigena` | Si el municipio tuvo circunscripción indígena |
| `votos_por_candidato` | Ratio votos / candidatos por partido |

El **Índice Herfindahl-Hirschman (HHI)** mide la concentración de un mercado (o de un sistema electoral). Se calcula sumando los cuadrados de las cuotas de participación. Va de 0 (competencia perfecta) a 10,000 (monopolio absoluto).

> **Pregunta guía:** ¿Podemos construir indicadores municipales que resuman la dinámica electoral local?

### 🔍 Interpretación — Paso 15

Las variables derivadas revelan patrones municipales interesantes:

- **HHI:** La mayoría de municipios tienen un HHI relativamente bajo (competencia dispersa), pero hay municipios con concentración muy alta donde uno o dos partidos dominan.
- **Relación inversa HHI vs. N° partidos:** Como era de esperar, a más partidos compitiendo, menor concentración del voto.
- **Municipios con circunscripción indígena:** Son una minoría, identificables como subgrupo diferenciado.

> Estas variables nos servirán como insumo para PCA y clustering en los próximos pasos.

## Paso 16 · Análisis multivariado y reducción de dimensionalidad (PCA)
🔴 Avanzado

Cuando tenemos muchas variables, es difícil visualizar sus relaciones simultáneamente. El **Análisis de Componentes Principales (PCA)** es una técnica que comprime múltiples variables en unas pocas «componentes» que capturan la mayor parte de la variación en los datos.

> **Analogía:** Imagina que describes un municipio con 6 indicadores (votos, partidos, concentración, etc.). PCA busca si existe un «eje principal» que resuma la mayor parte de esas diferencias. Quizás ese eje separa municipios grandes y competitivos de municipios pequeños y dominados por un solo partido.

> **Pregunta guía:** ¿Se pueden reducir nuestras variables municipales a 2 o 3 dimensiones sin perder mucha información? ¿Qué significan esas dimensiones en términos políticos?

### 🔍 Interpretación — Paso 16

El PCA comprime las 5 variables municipales en componentes interpretables:

- **PC1 (primer componente):** Captura principalmente el **tamaño del municipio** — votos totales, número de candidatos y votos por candidato apuntan en la misma dirección. Los municipios grandes están a un extremo, los pequeños al otro.
- **PC2 (segundo componente):** Captura la **concentración vs. dispersión** del voto — el HHI y el número de partidos apuntan en direcciones opuestas.

En términos políticos: la principal fuente de variación entre municipios es su **tamaño** (que determina cuántos votos, candidatos y partidos hay), y la segunda es la **estructura de competencia** (concentrada vs. fragmentada).

> Los municipios con circunscripción indígena (en naranja) tienden a agruparse en una zona del espacio, lo que sugiere que comparten características electorales distintivas.

## Paso 17 · Clustering exploratorio
🔴 Avanzado

El **clustering** busca agrupar municipios similares sin etiquetas previas. Usamos **K-Means**, un algoritmo que divide los datos en K grupos minimizando la distancia de cada punto al centro de su grupo.

Para elegir el número de clusters (K), usamos dos métodos:
- **Método del codo:** Graficar la "inercia" (suma de distancias al centro) vs. K. Buscamos el punto donde añadir más clusters deja de mejorar significativamente.
- **Coeficiente de silueta:** Mide qué tan bien asignado está cada punto a su cluster. Va de -1 (mal asignado) a +1 (perfectamente asignado).

> **Pregunta guía:** ¿Hay agrupaciones naturales de municipios según sus características electorales? ¿Son municipios de voto concentrado vs. fragmentado? ¿Grandes vs. pequeños?

### 🔍 Interpretación — Paso 17

El clustering revela agrupaciones naturales de municipios que tienen sentido político:

- **Municipios grandes y competitivos:** capitales de departamento con muchos votos, muchos partidos compitiendo y HHI bajo (competencia dispersa).
- **Municipios pequeños con alta concentración:** zonas rurales donde pocos partidos dominan, HHI alto, pocos candidatos.
- **Municipios intermedios:** la mayoría del país, con niveles moderados de competencia.

> **Reflexión:** Estos clusters reflejan la heterogeneidad territorial de Colombia. Un modelo electoral o de política pública que trate a todos los municipios por igual estaría ignorando estas diferencias estructurales. El clustering es una herramienta valiosa para segmentar análisis y diseñar intervenciones diferenciadas.

## Paso 18 · Exploración textual (pre-NLP)
🔴 Avanzado — ⚠️ NO APLICA A ESTE DATASET

Este paso analiza **texto libre** (discursos, actas, encuestas abiertas) mediante tokenización, frecuencia de términos y TF-IDF. Nuestro dataset no contiene texto libre — los nombres de partidos y candidatos son etiquetas fijas, no texto analizable con técnicas de NLP.

**¿Dónde sí aplicaría?**
- **Datos del SECOP** (contratación pública): los objetivos contractuales contienen texto descriptivo que puede analizarse con NLP.
- **Actas del Congreso:** El texto de debates legislativos permitiría modelado de tópicos para identificar las agendas de cada partido.
- **Redes sociales:** Tweets de candidatos durante la campaña permitirían análisis de sentimiento y posicionamiento ideológico.

> 💡 **Sugerencia:** Si deseas practicar exploración textual, el Portal de Datos Abiertos de Colombia (`datos.gov.co`) tiene datasets del SECOP con descripciones textuales de contratos públicos.

## Paso 19 · Validación de hipótesis y significancia estadística
🔴 Avanzado

En los pasos anteriores observamos patrones visuales. Ahora toca **validarlos estadísticamente**: ¿son reales o podrían ser producto del azar?

Vamos a testear tres hipótesis:

| # | Hipótesis | Test |
|---|-----------|------|
| H1 | Hay diferencia significativa en votos por candidato entre circunscripción Nacional e Indígena | Mann-Whitney U |
| H2 | La distribución de partidos es independiente del departamento | Chi-cuadrado |
| H3 | Los candidatos elegidos (curules=1) provienen de partidos con significativamente más votos | Mann-Whitney U |

Usamos tests **no paramétricos** (Mann-Whitney, Chi-cuadrado) porque, como vimos en el Paso 11, la distribución de votos no es normal.

**Conceptos clave:**
- **p-valor:** Probabilidad de observar un resultado tan extremo si la hipótesis nula fuera cierta. Si p < 0.05, rechazamos la hipótesis nula.
- **Tamaño del efecto:** Un p-valor bajo no implica un efecto grande. Con 300,000 registros, diferencias minúsculas pueden ser "significativas".

> **Pregunta guía:** ¿Nuestras observaciones visuales se sostienen ante tests formales?

### 🔍 Interpretación — Paso 19

Los tres tests confirman los patrones observados visualmente:

1. **H1 confirmada:** Hay diferencia significativa en votos entre circunscripciones. Pero el tamaño del efecto es relevante: la diferencia no es solo estadística sino sustantiva — las dos circunscripciones operan a escalas completamente distintas.

2. **H2 confirmada:** La distribución de votos por partido **no es independiente** del departamento. Esto era de esperar: los partidos tienen bastiones regionales. Centro Democrático domina en Antioquia, el Partido Liberal en la Costa Atlántica, etc.

3. **H3 confirmada:** Los candidatos que obtienen curul provienen de partidos con más votos totales a nivel nacional. Esto refleja que la **cifra repartidora** (el mecanismo de asignación de curules en Colombia) favorece a los partidos grandes.

> **Nota metodológica:** Con datasets de 300,000+ registros, prácticamente *cualquier* diferencia será estadísticamente significativa (p < 0.05). Por eso es crucial reportar también el **tamaño del efecto** y evaluar si la diferencia tiene **relevancia práctica**, no solo estadística.

## Paso 20 · Selección de variables (Feature Selection)
🔴 Avanzado

Si quisiéramos construir un modelo que prediga qué candidatos obtienen curul, necesitamos identificar las **variables más informativas**. Usaremos dos enfoques complementarios:

1. **Información mutua:** Mide cuánta información aporta cada variable sobre la variable objetivo (curules). No asume relaciones lineales.
2. **Importancia de variables con Random Forest:** Un modelo de ensamble que estima importancia basándose en cuánto mejora cada variable las predicciones.

> **Pregunta guía:** ¿Qué variables son las mejores predictoras de que un candidato obtenga curul?

### 🔍 Interpretación — Paso 20

Ambos métodos coinciden en que los **votos totales del candidato** y los **votos del partido a nivel nacional** son las variables más importantes para predecir la obtención de curul.

Esto tiene sentido con el sistema electoral colombiano:
- La **cifra repartidora** asigna curules usando los votos totales del partido.
- Dentro del partido, los candidatos con más votos individuales obtienen las curules asignadas.

> **Implicación para modelado:** Un modelo predictivo de asignación de curules debería centrarse en los votos (del candidato y del partido). Las variables geográficas y de circunscripción aportan menos al modelo, aunque son esenciales para el análisis descriptivo.

## Paso 21 · Evaluación de sesgos y representatividad
🔴 Avanzado

Antes de considerar estos datos como base para un modelo, debemos evaluar críticamente **a quién representan y a quién excluyen**. En ciencias políticas, esto no es un tecnicismo: un modelo entrenado con datos sesgados producirá predicciones que reproducen desigualdades.

### Sesgos identificados en este dataset:

**1. Ausencia de votos en blanco y nulos:**
El dataset contiene los votos asignados a partidos y candidatos, pero no registra el **voto en blanco**, el **voto nulo** ni la **abstención**. Esto significa que no podemos analizar el rechazo ciudadano a la oferta electoral ni la participación electoral real.

**2. Asimetría entre circunscripciones:**
La circunscripción Nacional tiene muchos más registros, partidos y votos que la Indígena. Cualquier modelo que se entrene con ambas sin ponderación adecuada estará dominado por los patrones de la Nacional.

**3. Posibles problemas de registro en zonas periféricas:**
Los departamentos con menor infraestructura administrativa (Vaupés, Guainía, Amazonas) podrían tener mayor proporción de subregistro o errores de captura. Los datos reflejan *lo registrado*, no necesariamente *lo ocurrido*.

**4. Candidatos con 0 votos en municipios donde no hicieron campaña:**
Que un candidato aparezca con 0 votos en un municipio no significa necesariamente que nadie votó por él allí — podría haber imprecisiones en la captura a nivel de mesa.

**5. Poblaciones no representadas:**
- La población afrocolombiana tiene circunscripción especial en la Cámara de Representantes, pero **no en el Senado** (excepto por la curul de paz). Este dataset no la captura.
- Las comunidades campesinas, desplazados y colombianos en el exterior están subrepresentados.

**6. Sesgo temporal:**
Los datos son de 2018. Las dinámicas electorales cambian entre elecciones. Conclusiones basadas en 2018 no son automáticamente extrapolables a 2022 o 2026.

> **Lección fundamental:** Los datos no son la realidad — son un *registro imperfecto* de la realidad. Cada dataset tiene ángulos muertos, y reconocerlos es lo que distingue un análisis riguroso de uno ingenuo.

## Paso 22 · Profiling automatizado como auditoría cruzada
🔴 Avanzado

Las herramientas de **EDA automatizado** (como `ydata-profiling` o `sweetviz`) generan reportes completos con un solo comando. No reemplazan el análisis manual, pero sirven como **auditoría cruzada** para detectar hallazgos que pudimos haber pasado por alto.

> **Pregunta guía:** ¿El reporte automatizado revela algo que no detectamos en los pasos previos?

### 🔍 Interpretación — Paso 22

El reporte automatizado sirve como **red de seguridad**: permite verificar que nuestro análisis manual no omitió patrones importantes. Algunas contribuciones típicas del profiling automático:
- Detecta automáticamente variables con **alta correlación** entre sí (multicolinealidad).
- Identifica variables con distribuciones **altamente sesgadas** (lo cual ya sabíamos).
- Señala variables con **alta cardinalidad** (muchas categorías únicas) que pueden ser problemáticas para modelado.
- Genera alertas de calidad que complementan nuestra revisión manual.

> **Nota:** Estas herramientas son un complemento, no un reemplazo. El valor del EDA manual que hicimos en los pasos anteriores es que incorpora **contexto sustantivo** que ningún algoritmo puede aportar.

## Paso 23 · Síntesis, documentación y transición al modelado
🔴 Avanzado

---

### 📋 Hallazgos clave del EDA

1. **Estructura del dataset:** ~300,000+ registros con dos tipos de filas (partido y candidato) que no deben mezclarse. La granularidad es municipio × partido/candidato.

2. **Distribución del voto:** Extremadamente sesgada (log-normal). La media es mucho mayor que la mediana. Unos pocos candidatos concentran la gran mayoría de los votos.

3. **Outliers legítimos:** Los candidatos más votados (Uribe, Mockus, Robledo, etc.) son outliers estadísticos pero no errores. Son la información más valiosa del dataset.

4. **Dos mundos electorales:** Las circunscripciones Nacional e Indígena operan con actores, escalas y dinámicas completamente distintas. Analizarlas conjuntamente enmascara diferencias fundamentales.

5. **Concentración territorial:** Los 5 departamentos más poblados dominan la votación. Los departamentos periféricos (Amazonía, Orinoquía) son cuantitativamente marginales.

6. **Bastiones regionales:** Los partidos no se distribuyen homogéneamente en el territorio. Hay preferencias regionales estadísticamente significativas.

7. **Predicción de curules:** Los votos totales del candidato y del partido son los mejores predictores de la obtención de curul, consistente con el sistema de cifra repartidora.

---

### ⚠️ Limitaciones

- No incluye votos en blanco, nulos ni abstención.
- Datos de un solo año (2018) — no permite análisis temporal.
- Posible subregistro en zonas rurales y periféricas.
- No captura la circunscripción afrocolombiana (solo existe para Cámara).

---

### 🔮 Variables candidatas para modelado

| Variable | Tipo | Uso potencial |
|----------|------|---------------|
| `votos` (agregados) | Continua | Variable de interés / predictor |
| `curules` | Binaria | Variable objetivo (clasificación) |
| `circunscripcion` | Categórica | Segmentación / feature |
| `departamento` | Categórica | Feature geográfico |
| `hhi_municipio` | Continua | Feature de estructura electoral |
| `n_partidos_municipio` | Discreta | Feature de competencia |
| `votos_por_candidato` | Continua | Feature de eficiencia partidista |

---

### 🔬 Preguntas abiertas para investigación futura

1. ¿Cómo se compara la concentración del voto en 2018 con elecciones anteriores y posteriores?
2. ¿Hay relación entre el HHI electoral municipal y variables socioeconómicas (pobreza, ruralidad, conflicto armado)?
3. ¿Los candidatos de la circunscripción indígena que obtienen curul tienen perfiles diferentes a los de la Nacional?
4. ¿Se puede predecir la probabilidad de obtener curul con un modelo logístico usando las variables derivadas?
5. ¿La fragmentación partidista varía sistemáticamente entre regiones afectadas por el conflicto armado y regiones pacíficas?

---

> **El EDA no "termina" aquí:** se reactiva cada vez que el modelo revele errores, residuos anómalos o preguntas nuevas. Es un ciclo vivo que se retroalimenta con el modelado y la interpretación.

---
*Fin del notebook — Curso de Analítica de Datos y Machine Learning en Ciencias Políticas — UdeA*